# 04 — Final Modelling & Production Forecasting

## Objective

This notebook contains the **final production modelling stage** of the retail sales forecasting project.

In the previous modelling stage (`03_Modelling.ipynb`), three forecasting strategies were evaluated:

1. One-Step-Ahead Forecasting
2. Rolling-Window Forecasting
3. Expanding-Window Forecasting

Based on the comparative evaluation across multiple forecasting folds, the **Rolling-Window Forecasting strategy** was selected as the final forecasting approach.

The purpose of this notebook is therefore not further model experimentation, but to convert the selected strategy into a **production-ready forecasting pipeline**.

## Final Forecasting Strategy

The final system will use:

> **XGBoost Regressor + 365-Day Rolling Training Window + Recursive 30-Day Forecasting**

### Rolling Training Window

For every forecasting period, the model is trained using the most recent **365 days** of historical data.

This allows the model to:

- Prioritize recent sales patterns
- Adapt to changing customer behaviour
- Capture recent promotional and seasonal effects
- Avoid giving excessive importance to very old observations

### Forecast Horizon

The production system will generate a forecast for the **next 30 days**.

The 30-day forecast is generated recursively while maintaining leakage safety.

For future dates:

- Actual historical sales are used whenever available.
- Previously generated predictions are used when actual future sales are unavailable.
- No actual future sales values are used to construct forecasting features.

## Why Rolling Window Was Selected

The Rolling-Window strategy was selected after comparing its performance with the Expanding-Window strategy acros

## Production Objective

The final objective is to create a forecasting system capable of answering:

> **"Given the latest available historical information, what are the expected sales for the next 30 days?"**

The resulting model and forecasting pipeline will later be integrated into an interactive web application where a business user or manager can generate and interpret sales forecasts.

In [1]:
import pandas as pd
import numpy as np

development_df = pd.read_csv(
    "../data/processed/development_df.csv"
)

development_df["Date"] = pd.to_datetime(
    development_df["Date"]
)

print("Development dataset loaded successfully.")
print("Shape:", development_df.shape)
print(
    "Date range:",
    development_df["Date"].min(),
    "to",
    development_df["Date"].max()
)

Development dataset loaded successfully.
Shape: (915744, 33)
Date range: 2013-01-31 00:00:00 to 2015-05-31 00:00:00


In [2]:
ROLLING_TRAIN_DAYS = 365

final_train_end = development_df["Date"].max()

final_train_start = (
    final_train_end
    - pd.Timedelta(days=ROLLING_TRAIN_DAYS - 1)
)

print("Final rolling-window training period")
print("------------------------------------")
print("Training start:", final_train_start)
print("Training end  :", final_train_end)

Final rolling-window training period
------------------------------------
Training start: 2014-06-01 00:00:00
Training end  : 2015-05-31 00:00:00


In [3]:
#Remember, we already created all of our lag/rolling features in development_df.

#Therefore DO NOT recreate them here.

#We simply slice the latest 365 days.

final_train_df = development_df[
    (development_df["Date"] >= final_train_start) &
    (development_df["Date"] <= final_train_end)
].copy()

final_train_df = final_train_df.sort_values(
    ["Store", "Date"]
).reset_index(drop=True)

print("Final training rows:", len(final_train_df))
print(
    "Date range:",
    final_train_df["Date"].min(),
    "->",
    final_train_df["Date"].max()
)

Final training rows: 373855
Date range: 2014-06-01 00:00:00 -> 2015-05-31 00:00:00


In [4]:
feature_cols = development_df.columns.tolist()

if "Sales" in feature_cols:
    feature_cols.remove("Sales")

if "Date" in feature_cols:
    feature_cols.remove("Date")

if "Customers" in feature_cols:
    feature_cols.remove("Customers")

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 31
['Store', 'DayOfWeek', 'Open', 'Promo', 'SchoolHoliday', 'CompetitionDistance', 'Promo2', 'Year', 'Month', 'Day', 'WeekOfYear', 'CompetitionOpenMonths', 'Promo2Active', 'IsStateHoliday', 'StoreType_a', 'StoreType_b', 'StoreType_c', 'StoreType_d', 'Assortment_a', 'Assortment_b', 'Assortment_c', 'StateHoliday_0', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Rolling_7', 'Sales_Rolling_14', 'Sales_Rolling_30']


In [5]:
X_final = final_train_df[feature_cols]
y_final = final_train_df["Sales"]

print("X_final shape:", X_final.shape)
print("y_final shape:", y_final.shape)
print("Number of features:", len(feature_cols))

X_final shape: (373855, 31)
y_final shape: (373855,)
Number of features: 31


In [6]:
from xgboost import XGBRegressor

final_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [7]:
print("Training final rolling-window model...")
print(
    f"Training period: "
    f"{final_train_start.date()} → {final_train_end.date()}"
)
print(f"Training rows: {len(X_final):,}")
print(f"Features: {len(feature_cols)}")

final_model.fit(
    X_final,
    y_final,
    verbose=False
)

print("\nFinal rolling-window model trained successfully.")

Training final rolling-window model...
Training period: 2014-06-01 → 2015-05-31
Training rows: 373,855
Features: 31

Final rolling-window model trained successfully.


In [8]:
print(final_model)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=-1, num_parallel_tree=None, ...)


In [9]:
print("Number of trees:", final_model.n_estimators)
print("Number of features:", len(feature_cols))

Number of trees: 500
Number of features: 31


In [10]:
feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": final_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(15)

,Feature,Importance
2,Open,0.705829
3,Promo,0.051170
13,IsStateHoliday,0.036267
30,Sales_Rolling_30,0.035663
27,Sales_Lag_14,0.028190
22,StateHoliday_a,0.026286
24,StateHoliday_c,0.021250
21,StateHoliday_0,0.013791
1,DayOfWeek,0.013161
15,StoreType_b,0.011689


In [11]:
from pathlib import Path
import joblib

# Project root = one level above the notebooks folder
PROJECT_ROOT = Path.cwd().parent

# Models folder
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save final XGBoost model
joblib.dump(
    final_model,
    MODELS_DIR / "final_rolling_xgb_model.pkl"
)

# Save feature list
joblib.dump(
    feature_cols,
    MODELS_DIR / "feature_cols.pkl"
)

print("Models saved successfully!")
print("Model:", MODELS_DIR / "final_rolling_xgb_model.pkl")
print("Features:", MODELS_DIR / "feature_cols.pkl")

Models saved successfully!
Model: e:\Demand-Forecasting_System\models\final_rolling_xgb_model.pkl
Features: e:\Demand-Forecasting_System\models\feature_cols.pkl


In [12]:
import os

print("Processed files:")
for file in os.listdir("../data/processed"):
    print(file)

Processed files:
development_df.csv
model_data.csv
rolling_backtest_results.csv


In [13]:
import pandas as pd

development_check = pd.read_csv(
    "../data/processed/development_df.csv"
)

model_data_check = pd.read_csv(
    "../data/processed/model_data.csv"
)

development_check["Date"] = pd.to_datetime(
    development_check["Date"]
)

model_data_check["Date"] = pd.to_datetime(
    model_data_check["Date"]
)

print("=" * 60)
print("DEVELOPMENT DATASET")
print("=" * 60)

print("Shape:", development_check.shape)
print(
    "Date range:",
    development_check["Date"].min(),
    "→",
    development_check["Date"].max()
)

print()

print("=" * 60)
print("MODEL DATASET")
print("=" * 60)

print("Shape:", model_data_check.shape)
print(
    "Date range:",
    model_data_check["Date"].min(),
    "→",
    model_data_check["Date"].max()
)

DEVELOPMENT DATASET
Shape: (915744, 33)
Date range: 2013-01-31 00:00:00 → 2015-05-31 00:00:00

MODEL DATASET
Shape: (983759, 33)
Date range: 2013-01-31 00:00:00 → 2015-07-31 00:00:00


In [14]:

# LOAD FULL MODEL DATA


model_data = pd.read_csv(
    "../data/processed/model_data.csv"
)

model_data["Date"] = pd.to_datetime(
    model_data["Date"]
)

print("Model data shape:", model_data.shape)
print(
    "Date range:",
    model_data["Date"].min(),
    "→",
    model_data["Date"].max()
)

Model data shape: (983759, 33)
Date range: 2013-01-31 00:00:00 → 2015-07-31 00:00:00


In [15]:

# FINAL HELD-OUT TEST PERIOD

forecast_start = pd.Timestamp("2015-06-01")
forecast_end = pd.Timestamp("2015-06-30")

test_df = model_data[
    (model_data["Date"] >= forecast_start) &
    (model_data["Date"] <= forecast_end)
].copy()

print("=" * 60)
print("FINAL HELD-OUT TEST DATA")
print("=" * 60)

print("Forecast period:",
      forecast_start.date(),
      "→",
      forecast_end.date())

print("Rows:", len(test_df))
print("Stores:", test_df["Store"].nunique())
print("Date range:",
      test_df["Date"].min(),
      "→",
      test_df["Date"].max())

FINAL HELD-OUT TEST DATA
Forecast period: 2015-06-01 → 2015-06-30
Rows: 33450
Stores: 1115
Date range: 2015-06-01 00:00:00 → 2015-06-30 00:00:00


In [16]:
# historical_df must be the SAME window the final model was trained on,
# since that's what determines lag_1 / lag_7 / lag_14 / rolling means
# for the first few days of the forecast horizon.

final_train_start = pd.Timestamp("2014-06-01")
final_train_end = pd.Timestamp("2015-05-31")

final_historical_df = development_df[
    (development_df["Date"] >= final_train_start) &
    (development_df["Date"] <= final_train_end)
].copy()

print("Historical window for lags:", final_train_start.date(), "→", final_train_end.date())
print("Historical rows:", len(final_historical_df))

Historical window for lags: 2014-06-01 → 2015-05-31
Historical rows: 373855


In [17]:
import numpy as np
import pandas as pd


def recursive_forecast(model, historical_df, future_df, feature_cols):
    """
    Leakage-safe recursive multi-step forecasting, batched by date.
    """
    sales_pivot = historical_df.pivot(index="Date", columns="Store", values="Sales")

    all_stores = future_df["Store"].unique()
    missing_stores = set(all_stores) - set(sales_pivot.columns)
    for s in missing_stores:
        sales_pivot[s] = np.nan
    sales_pivot = sales_pivot.sort_index(axis=1)

    forecast_dates = sorted(future_df["Date"].unique())

    def get_lag(date, offset_days, stores):
        target_date = date - pd.Timedelta(days=offset_days)
        if target_date in sales_pivot.index:
            return sales_pivot.loc[target_date, stores].values
        return np.full(len(stores), np.nan)

    def get_rolling_mean(date, window, stores):
        window_dates = [date - pd.Timedelta(days=i) for i in range(1, window + 1)]
        window_dates = [d for d in window_dates if d in sales_pivot.index]
        if not window_dates:
            return np.full(len(stores), np.nan)
        sub = sales_pivot.loc[window_dates, stores]
        return sub.mean(axis=0, skipna=True).values

    prediction_frames = []

    for date in forecast_dates:
        day_rows = future_df[future_df["Date"] == date].copy()
        stores_today = day_rows["Store"].values

        day_rows["Sales_Lag_1"] = get_lag(date, 1, stores_today)
        day_rows["Sales_Lag_7"] = get_lag(date, 7, stores_today)
        day_rows["Sales_Lag_14"] = get_lag(date, 14, stores_today)
        day_rows["Sales_Rolling_7"] = get_rolling_mean(date, 7, stores_today)
        day_rows["Sales_Rolling_14"] = get_rolling_mean(date, 14, stores_today)
        day_rows["Sales_Rolling_30"] = get_rolling_mean(date, 30, stores_today)

        X_day = day_rows[feature_cols]
        preds_today = model.predict(X_day)

        if date not in sales_pivot.index:
            sales_pivot.loc[date] = np.nan
        sales_pivot.loc[date, stores_today] = preds_today

        day_rows["Predicted_Sales"] = preds_today
        prediction_frames.append(day_rows[["Store", "Date", "Predicted_Sales"]])

    all_preds = pd.concat(prediction_frames, ignore_index=True)
    aligned = future_df[["Store", "Date"]].merge(all_preds, on=["Store", "Date"], how="left")
    return aligned["Predicted_Sales"].values

In [18]:


y_pred_final = recursive_forecast(
    model=final_model,
    historical_df=final_historical_df,
    future_df=test_df,
    feature_cols=feature_cols
)

print("Forecast generated for", len(y_pred_final), "rows")

Forecast generated for 33450 rows


here the problem was , for close store days, the actual sales were 0, but prediction shows both positive and negative values, so we chip both the values to 0, based on if the store is closed, then actual and perdicted sales both will be 0

In [19]:

# FINAL FORECAST TABLE


forecast_table = test_df[
    ["Store", "Date", "Sales", "Open"]
].copy()

# Rename actual sales
forecast_table = forecast_table.rename(
    columns={"Sales": "Actual_Sales"}
)

# Add model predictions
forecast_table["Predicted_Sales"] = y_pred_final

# Business rule:
# If the store is closed, both actual and predicted sales = 0


forecast_table.loc[
    forecast_table["Open"] == 0,
    "Actual_Sales"
] = 0

forecast_table.loc[
    forecast_table["Open"] == 0,
    "Predicted_Sales"
] = 0

# Sales can never be negative
forecast_table["Predicted_Sales"] = (
    forecast_table["Predicted_Sales"].clip(lower=0)
)

# Keep only the final required columns
forecast_table = forecast_table[
    ["Store", "Date", "Actual_Sales", "Predicted_Sales"]
]

print("Final forecast table created successfully.")
print("Rows:", len(forecast_table))

forecast_table.head(10)

Final forecast table created successfully.
Rows: 33450


,Store,Date,Actual_Sales,Predicted_Sales
851,1,2015-06-01,5774,6523.941406
852,1,2015-06-02,5450,5805.670410
853,1,2015-06-03,5809,5424.985840
854,1,2015-06-04,0,0.000000
855,1,2015-06-05,5384,6609.137695
856,1,2015-06-06,4183,4777.521973
857,1,2015-06-07,0,0.000000
858,1,2015-06-08,4071,4005.109619
859,1,2015-06-09,4102,3739.809326
860,1,2015-06-10,3591,3994.453125


In [20]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_actual_final = forecast_table["Actual_Sales"].values
y_pred_final_arr = forecast_table["Predicted_Sales"].values

final_mae = mean_absolute_error(y_actual_final, y_pred_final_arr)
final_rmse = np.sqrt(mean_squared_error(y_actual_final, y_pred_final_arr))
final_wape = np.sum(np.abs(y_actual_final - y_pred_final_arr)) / np.sum(np.abs(y_actual_final)) * 100
final_r2 = r2_score(y_actual_final, y_pred_final_arr)

print("=" * 50)
print("FINAL TEST EVALUATION (June 2015, never touched before now)")
print("=" * 50)
print(f"MAE  : {final_mae:.2f}")
print(f"RMSE : {final_rmse:.2f}")
print(f"WAPE : {final_wape:.2f}%")
print(f"R²   : {final_r2:.4f}")

FINAL TEST EVALUATION (June 2015, never touched before now)
MAE  : 577.35
RMSE : 881.19
WAPE : 9.31%
R²   : 0.9513


In [21]:
import pandas as pd

rolling_df = pd.read_csv(
    "../data/processed/rolling_backtest_results.csv"
)

print("Rolling backtest results loaded successfully.")
print("Shape:", rolling_df.shape)
print(rolling_df.head())

Rolling backtest results loaded successfully.
Shape: (16, 9)
   Fold Train_Start   Train_End Forecast_Start Forecast_End         MAE  \
0     1  2013-01-31  2014-01-30     2014-01-31   2014-03-01  527.404724   
1     2  2013-03-02  2014-03-01     2014-03-02   2014-03-31  569.541382   
2     3  2013-04-01  2014-03-31     2014-04-01   2014-04-30  858.624573   
3     4  2013-05-01  2014-04-30     2014-05-01   2014-05-30  584.653503   
4     5  2013-05-31  2014-05-30     2014-05-31   2014-06-29  570.588318   

          RMSE       WAPE        R2  
0   770.575678   9.096021  0.952058  
1   965.103557  10.246365  0.931957  
2  1305.364652  14.763162  0.895995  
3   862.822983  10.412856  0.949786  
4   866.335313  10.394444  0.951944  


In [22]:


print("Dev backtest average (16 folds):")
print(f"  WAPE: {rolling_df['WAPE'].mean():.2f}%  |  R²: {rolling_df['R2'].mean():.4f}")
print("Final held-out test (June 2015):")
print(f"  WAPE: {final_wape:.2f}%  |  R²: {final_r2:.4f}")

Dev backtest average (16 folds):
  WAPE: 11.57%  |  R²: 0.9252
Final held-out test (June 2015):
  WAPE: 9.31%  |  R²: 0.9513
